In [5]:
"""
Baza transakcji M&A i rund finansowania w segmencie ladowania EV w Europie.

Zebrane z publicznie dostepnych zrodel: komunikaty prasowe spolek, prasa
branzowa (EVBoosters, GreenTechMedia, Autocar, FleetNews, ChargedEVs),
Wikipedia. Kazda pozycja ma zrodlo i date.

MNOZNIKI: kolumny EV/kW i EV/punkt zostaly USUNIETE - sprawdzono szeroko
i takie mnozniki nie sa publicznie ujawniane dla praktycznie zadnej z tych
transakcji (zrodla podaja kwote ALBO wielkosc sieci, rzadko oba naraz i z
tego samego momentu). Zamiast tego arkusz "Benchmarki wyceny" zawiera
mnozniki EV/EBITDA - jedyne, ktore branza faktycznie publikuje jako
zagregowane wskazniki.

UWAGA: "EV" w EV/EBITDA to Enterprise Value (wartosc przedsiebiorstwa),
nie Electric Vehicle.

Wymaga: brak (dane sa statyczne, wpisane w kodzie ponizej)
Wynik: baza_transakcji_ev.xlsx
"""

import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import CellIsRule

TRANSAKCJE = [
    # ---------- RUNDY FINANSOWANIA ----------
    {
        "Spolka": "IONITY", "Kraj": "Niemcy", "Data": "2025-05-15",
        "Typ": "Green Loan Facility", "Kwota_EUR_mln": 600,
        "Inwestorzy_kredytodawcy": "ABN AMRO, BNP Paribas, Credit Agricole CIB, ING, KfW IPEX-Bank, LBBW, MUFG, NordLB, Rabobank + OEM-y (BMW, Mercedes, VW, Ford, Hyundai), GIP (BlackRock)",
        "Zrodlo": "https://www.ionity.eu/ionity/press-releases/ionity-secures-record-financing-to-expand-critical-infrastructure-for-europes-electric-future",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "IONITY", "Kraj": "Niemcy", "Data": "2021 (rok, dokladna data nieujawniona)",
        "Typ": "Equity (runda inwestycyjna)", "Kwota_EUR_mln": 700,
        "Inwestorzy_kredytodawcy": "Global Infrastructure Partners (GIP, czesc BlackRock) + dotychczasowi udzialowcy OEM",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Milence", "Kraj": "Holandia", "Data": "2022 (rok, dokladna data nieujawniona)",
        "Typ": "Equity (utworzenie JV)", "Kwota_EUR_mln": 500,
        "Inwestorzy_kredytodawcy": "Daimler Truck, TRATON Group, Volvo Group",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Milence", "Kraj": "Holandia", "Data": "2026 (H1, dokladna data nieujawniona)",
        "Typ": "Financing Facility (dlug)", "Kwota_EUR_mln": 120,
        "Inwestorzy_kredytodawcy": "nieujawnieni w zrodle (finansowanie wspierane przez Daimler Truck, TRATON, Volvo Group jako udzialowcow)",
        "Zrodlo": "https://evboosters.com/ev-charging-news/what-8-cpo-financing-deals-worth-e2-2-billion-since-2025-reveal-about-europes-ev-charging-future/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Allego", "Kraj": "Holandia", "Data": "2022 (rok, dokladna data nieujawniona)",
        "Typ": "Debt Facility", "Kwota_EUR_mln": 400,
        "Inwestorzy_kredytodawcy": "Societe Generale, Banco Santander",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Allego", "Kraj": "Holandia", "Data": "2024 (rok, dokladna data nieujawniona)",
        "Typ": "Convertible Loan", "Kwota_EUR_mln": 310,
        "Inwestorzy_kredytodawcy": "Meridiam (wiekszosciowy udzialowiec)",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Electra", "Kraj": "Francja", "Data": "2024 (rok, dokladna data nieujawniona)",
        "Typ": "Equity (Series B)", "Kwota_EUR_mln": 304,
        "Inwestorzy_kredytodawcy": "PGGM, Bpifrance, Eurazeo, SNCF, Serena, RIVE",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Electra", "Kraj": "Francja", "Data": "2026 (H1, dokladna data nieujawniona)",
        "Typ": "Green Loan Facility", "Kwota_EUR_mln": 433,
        "Inwestorzy_kredytodawcy": "nieujawnieni w zrodle (laczne finansowanie spolki od zalozenia przekroczylo 1 mld EUR)",
        "Zrodlo": "https://evboosters.com/ev-charging-news/what-8-cpo-financing-deals-worth-e2-2-billion-since-2025-reveal-about-europes-ev-charging-future/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Driveco", "Kraj": "Francja", "Data": "2023 (rok, dokladna data nieujawniona)",
        "Typ": "Equity", "Kwota_EUR_mln": 250,
        "Inwestorzy_kredytodawcy": "APG, Mirova, Corsica Sole",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Zunder", "Kraj": "Hiszpania", "Data": "2024 (rok, dokladna data nieujawniona)",
        "Typ": "Loan", "Kwota_EUR_mln": 225,
        "Inwestorzy_kredytodawcy": "Santander",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Recharge", "Kraj": "Norwegia", "Data": "2024 (rok, dokladna data nieujawniona)",
        "Typ": "Green Debt", "Kwota_EUR_mln": 180,
        "Inwestorzy_kredytodawcy": "KfW IPEX-Bank + 3 dodatkowe banki finansowania projektowego",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Powerdot", "Kraj": "Portugalia", "Data": "2024 (rok, dokladna data nieujawniona)",
        "Typ": "Green Debt", "Kwota_EUR_mln": 165,
        "Inwestorzy_kredytodawcy": "ABN AMRO, BNP Paribas, ING, MUFG, Santander, Societe Generale",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Powerdot", "Kraj": "Portugalia", "Data": "2024 (rok, dokladna data nieujawniona)",
        "Typ": "Equity", "Kwota_EUR_mln": 100,
        "Inwestorzy_kredytodawcy": "Antin Infrastructure Partners, Arie Group (dotychczasowi udzialowcy)",
        "Zrodlo": "https://evboosters.com/ev-charging-news/inside-europes-biggest-ev-charging-financing-deals-2022-2025-how-institutional-investors-power-cpo-growth-with-billions/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Believ", "Kraj": "Wielka Brytania", "Data": "2025-2026 (dokladna data nieujawniona)",
        "Typ": "Investment Facility", "Kwota_EUR_mln": 350,
        "Inwestorzy_kredytodawcy": "nieujawnieni w zrodle (kwota oryginalna: 300 mln GBP)",
        "Zrodlo": "https://evboosters.com/ev-charging-news/what-8-cpo-financing-deals-worth-e2-2-billion-since-2025-reveal-about-europes-ev-charging-future/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "InstaVolt", "Kraj": "Wielka Brytania", "Data": "2025-2026 (dokladna data nieujawniona)",
        "Typ": "Debt Financing", "Kwota_EUR_mln": 293,
        "Inwestorzy_kredytodawcy": "nieujawnieni w zrodle (kwota oryginalna: 250 mln GBP)",
        "Zrodlo": "https://evboosters.com/ev-charging-news/what-8-cpo-financing-deals-worth-e2-2-billion-since-2025-reveal-about-europes-ev-charging-future/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Fastned", "Kraj": "Holandia (finansowanie: Belgia, Szwajcaria)", "Data": "2025-2026 (dokladna data nieujawniona)",
        "Typ": "Green Financing", "Kwota_EUR_mln": 200,
        "Inwestorzy_kredytodawcy": "nieujawnieni w zrodle",
        "Zrodlo": "https://evboosters.com/ev-charging-news/what-8-cpo-financing-deals-worth-e2-2-billion-since-2025-reveal-about-europes-ev-charging-future/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "GreenWay", "Kraj": "Polska/Slowacja/Chorwacja", "Data": "2025-2026 (dokladna data nieujawniona)",
        "Typ": "Green Debt Financing", "Kwota_EUR_mln": 138,
        "Inwestorzy_kredytodawcy": "nieujawnieni w zrodle",
        "Zrodlo": "https://evboosters.com/ev-charging-news/what-8-cpo-financing-deals-worth-e2-2-billion-since-2025-reveal-about-europes-ev-charging-future/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Osprey Charging", "Kraj": "Wielka Brytania", "Data": "2025-2026 (dokladna data nieujawniona)",
        "Typ": "Senior Debt Facilities", "Kwota_EUR_mln": 129,
        "Inwestorzy_kredytodawcy": "nieujawnieni w zrodle (kwota oryginalna: 110 mln GBP)",
        "Zrodlo": "https://evboosters.com/ev-charging-news/what-8-cpo-financing-deals-worth-e2-2-billion-since-2025-reveal-about-europes-ev-charging-future/",
        "Data_dostepu": "2026-08-08",
    },
    # ---------- M&A (PRZEJECIA / FUZJE) ----------
    {
        "Spolka": "Chargemaster (przejeta przez BP)", "Kraj": "Wielka Brytania", "Data": "2018-06",
        "Typ": "Przejecie (100%)", "Kwota_EUR_mln": 147,  # ok. 130 mln GBP wg kursu historycznego
        "Inwestorzy_kredytodawcy": "BP (nabywca)",
        "Mnoznik_EV_na_kW": None, "Mnoznik_EV_na_punkt": 19286,  # 130mln GBP / ok. 6750 punktow (srednia 6500-7000)
        "Zrodlo": "https://www.greentechmedia.com/articles/read/bp-buys-chargemaster-britains-largest-ev-charging-company",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Pod Point (53% udzialow, przejete przez EDF + Legal & General)", "Kraj": "Wielka Brytania", "Data": "2020-02",
        "Typ": "Przejecie wiekszosciowego pakietu (53%)", "Kwota_EUR_mln": 125,  # ok. 110 mln GBP
        "Inwestorzy_kredytodawcy": "EDF Energy, Legal & General Capital (nabywcy)",
        "Mnoznik_EV_na_kW": None, "Mnoznik_EV_na_punkt": None,  # zrodlo podaje 62000+6600 punktow z PO okresu przejecia (2021), nie w momencie transakcji - za duza niepewnosc
        "Zrodlo": "https://www.fleetnews.co.uk/news/fleet-industry-news/2020/02/14/edf-acquires-ev-chargepoint-provider-pod-point-in-110-million-deal",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "ubitricity (przejeta przez Shell, 100%)", "Kraj": "Niemcy/Wielka Brytania", "Data": "2021-01",
        "Typ": "Przejecie (100%)", "Kwota_EUR_mln": None,  # nieujawniona
        "Inwestorzy_kredytodawcy": "Shell (nabywca)",
        "Zrodlo": "https://chargedevs.com/newswire/oil-companies-buying-up-ev-charging-networks-shell-acquires-ubitricity/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Volta (przejeta przez Shell)", "Kraj": "USA", "Data": "2023 (dokladna data nieujawniona)",
        "Typ": "Przejecie (100%, gotowka)", "Kwota_EUR_mln": 155,  # ok. 169 mln USD
        "Inwestorzy_kredytodawcy": "Shell (nabywca)",
        "Zrodlo": "https://www.energymonitor.ai/tech/networks-grids/how-oil-majors-penetrated-the-ev-charging-market/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "ABL (przejeta przez Wallbox)", "Kraj": "Niemcy", "Data": "2023-10-18",
        "Typ": "Przejecie aktywow i operacji", "Kwota_EUR_mln": 15,
        "Inwestorzy_kredytodawcy": "Wallbox (nabywca)",
        "Zrodlo": "https://www.businesswire.com/newsroom/subject/merger-acquisition?page=1078",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Mer (fuzja z Eviny Fast Charging)", "Kraj": "Norwegia/Skandynawia", "Data": "2026-07-09",
        "Typ": "Fuzja (Statkraft obejmuje 43% nowego podmiotu)", "Kwota_EUR_mln": None,
        "Inwestorzy_kredytodawcy": "Eviny Fast Charging, Statkraft (43% udzialow w nowym podmiocie)",
        "Zrodlo": "https://app.fundz.net/acquisitions/acquires-9b57",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Mer (siec UK, przejeta przez Be.EV)", "Kraj": "Wielka Brytania", "Data": "2026-02 (dokladna data nieujawniona)",
        "Typ": "Przejecie sieci regionalnej (450 lokalizacji, 1600 stanowisk)", "Kwota_EUR_mln": None,
        "Inwestorzy_kredytodawcy": "Be.EV (nabywca)",
        "Zrodlo": "https://www.flexecharge.com/resources/blogs/february-2026-strategic-acquisitions-and-urban-expansion-drive-europes-ev-charging-evolution",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "Blue Charge (przejeta przez TotalEnergies)", "Kraj": "Singapur (nabywca: Francja)", "Data": "2023-2025 (dokladna data nieujawniona)",
        "Typ": "Przejecie (wejscie na nowy rynek geograficzny)", "Kwota_EUR_mln": None,
        "Inwestorzy_kredytodawcy": "TotalEnergies (nabywca)",
        "Zrodlo": "https://apricum-group.com/the-ev-charging-market-is-ripe-for-consolidation/",
        "Data_dostepu": "2026-08-08",
    },
    {
        "Spolka": "eeMobility (pakiet wiekszosciowy, przejety przez Mer/Statkraft)", "Kraj": "Niemcy (nabywca: Norwegia)", "Data": "2023-2025 (dokladna data nieujawniona)",
        "Typ": "Przejecie wiekszosciowego pakietu (wejscie na rynek niemiecki)", "Kwota_EUR_mln": None,
        "Inwestorzy_kredytodawcy": "Mer (grupa Statkraft) - nabywca",
        "Zrodlo": "https://apricum-group.com/the-ev-charging-market-is-ripe-for-consolidation/",
        "Data_dostepu": "2026-08-08",
    },
]

print(f"Liczba pozycji w bazie: {len(TRANSAKCJE)}")

df = pd.DataFrame(TRANSAKCJE)
df = df[["Spolka", "Kraj", "Data", "Typ", "Kwota_EUR_mln",
         "Inwestorzy_kredytodawcy", "Zrodlo", "Data_dostepu"]]
df.columns = ["Spolka / cel transakcji", "Kraj", "Data", "Typ transakcji",
              "Kwota (mln EUR)", "Inwestorzy / kredytodawcy",
              "Zrodlo", "Data dostepu do zrodla"]

CZCIONKA = "Segoe UI"
KOLOR_CIEMNY = "1B365D"

# ---------- Benchmarki wyceny (EV/EBITDA) ----------
# UWAGA: "EV" = Enterprise Value (wartosc przedsiebiorstwa), NIE Electric Vehicle.
BENCHMARKI = [
    ("Wykonawcy instalacji", "3,0 - 5,0x EBITDA",
     "Najnizsze mnozniki - model uslugowy, niskie marze (8-12%), brak powtarzalnych przychodow",
     "https://ctacquisitions.com/how-to-sell-an-ev-charging-business/", "2026-08-08"),
    ("Instalacja + serwis (regionalnie)", "4,0 - 6,0x EBITDA",
     "Dodanie serwisu podnosi mnoznik dzieki czesciowo powtarzalnym przychodom",
     "https://ctacquisitions.com/how-to-sell-an-ev-charging-business/", "2026-08-08"),
    ("Operatorzy sieci (CPO) z istotna utylizacja", "6,0 - 9,0x EBITDA",
     "Kluczowy czynnik wyceny to UTYLIZACJA, nie sama liczba punktow",
     "https://ctacquisitions.com/how-to-sell-an-ev-charging-business/", "2026-08-08"),
    ("Sieci hubow DC (DCFC)", "7,0 - 10,0x EBITDA",
     "Premia za szybkie ladowanie i jakosc lokalizacji",
     "https://ctacquisitions.com/how-to-sell-an-ev-charging-business/", "2026-08-08"),
    ("Platformy premium o duzej skali", "8,0 - 12,0x+ EBITDA",
     "Najwyzsze mnozniki - powtarzalne przychody, oprogramowanie, subskrypcje",
     "https://ctacquisitions.com/how-to-sell-an-ev-charging-business/", "2026-08-08"),
    ("Rynek szwajcarski - mnozniki TRANSAKCYJNE", "6,5 - 9,5x EBITDA",
     "Niezalezne zrodlo potwierdzajace zakres; wyraznie powyzej zakresu statystycznego 5,0-7,0x",
     "https://valindex.ch/en/valuation/ev-charging-infrastructure/", "2026-08-08"),
    ("Rynek szwajcarski - zakres STATYSTYCZNY", "5,0 - 7,0x EBITDA",
     "Zakres bazowy (nie transakcyjny) - roznica pokazuje premie placona w realnych transakcjach",
     "https://valindex.ch/en/valuation/ev-charging-infrastructure/", "2026-08-08"),
]

df_bench = pd.DataFrame(BENCHMARKI, columns=[
    "Profil biznesowy", "Mnoznik EV/EBITDA", "Komentarz", "Zrodlo", "Data dostepu"])

with pd.ExcelWriter("baza_transakcji_ev.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Transakcje", index=False)
    df_bench.to_excel(writer, sheet_name="Benchmarki wyceny", index=False)
    ws = writer.sheets["Transakcje"]

    for kom in ws[1]:
        kom.font = Font(name=CZCIONKA, bold=True, color="FFFFFF", size=11)
        kom.fill = PatternFill("solid", fgColor=KOLOR_CIEMNY)
        kom.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32

    for wiersz in ws.iter_rows(min_row=2):
        for kom in wiersz:
            kom.font = Font(name=CZCIONKA, size=10)
            kom.alignment = Alignment(vertical="top", wrap_text=True)
        wiersz[0].alignment = Alignment(vertical="top", wrap_text=True)

    ws.freeze_panes = "A2"
    szerokosci = [42, 20, 22, 26, 14, 48, 40, 16]
    for i, szer in enumerate(szerokosci, start=1):
        ws.column_dimensions[get_column_letter(i)].width = szer

    ws_b = writer.sheets["Benchmarki wyceny"]
    for kom in ws_b[1]:
        kom.font = Font(name=CZCIONKA, bold=True, color="FFFFFF", size=11)
        kom.fill = PatternFill("solid", fgColor=KOLOR_CIEMNY)
        kom.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws_b.row_dimensions[1].height = 32
    for wiersz in ws_b.iter_rows(min_row=2):
        for kom in wiersz:
            kom.font = Font(name=CZCIONKA, size=10)
            kom.alignment = Alignment(vertical="top", wrap_text=True)
    ws_b.freeze_panes = "A2"
    for i, szer in enumerate([40, 22, 62, 40, 16], start=1):
        ws_b.column_dimensions[get_column_letter(i)].width = szer

print("Zapisano: baza_transakcji_ev.xlsx")

Liczba pozycji w bazie: 27
Zapisano: baza_transakcji_ev.xlsx
